In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
english_sentences = [
    "hello",
    "how are you",
    "good morning",
    "good night",
    "thank you",
    "i love machine learning",
    "where is the station",
    "what is your name",
    "i am a student",
    "see you later"
]

german_sentences = [
    "hallo",
    "wie geht es dir",
    "guten morgen",
    "gute nacht",
    "danke",
    "ich liebe maschinelles lernen",
    "wo ist der bahnhof",
    "wie heisst du",
    "ich bin ein student",
    "bis später"
]

german_sentences = ["startseq " + sentence + " endseq" for sentence in german_sentences]

print("English sample:", english_sentences[0])
print("German sample:", german_sentences[0])

English sample: hello
German sample: startseq hallo endseq


In [3]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

ger_tokenizer = Tokenizer()
ger_tokenizer.fit_on_texts(german_sentences)

eng_vocab_size = len(eng_tokenizer.word_index) + 1
ger_vocab_size = len(ger_tokenizer.word_index) + 1

print("English vocabulary size:", eng_vocab_size)
print("German vocabulary size:", ger_vocab_size)

English vocabulary size: 25
German vocabulary size: 28


In [4]:
max_eng_len = max(len(sentence.split()) for sentence in english_sentences)
max_ger_len = max(len(sentence.split()) for sentence in german_sentences)

encoder_input = eng_tokenizer.texts_to_sequences(english_sentences)
decoder_input = ger_tokenizer.texts_to_sequences(german_sentences)

encoder_input = pad_sequences(encoder_input, maxlen=max_eng_len, padding="post")
decoder_input = pad_sequences(decoder_input, maxlen=max_ger_len, padding="post")

decoder_output = np.zeros_like(decoder_input)
decoder_output[:, :-1] = decoder_input[:, 1:]

print("Encoder input shape:", encoder_input.shape)
print("Decoder input shape:", decoder_input.shape)
print("Decoder output shape:", decoder_output.shape)

Encoder input shape: (10, 4)
Decoder input shape: (10, 6)
Decoder output shape: (10, 6)


In [5]:
embedding_dim = 64
latent_dim = 128

encoder_inputs = tf.keras.layers.Input(shape=(max_eng_len,))
encoder_embedding = tf.keras.layers.Embedding(
    eng_vocab_size,
    embedding_dim
)(encoder_inputs)

encoder_lstm, state_h, state_c = tf.keras.layers.LSTM(
    latent_dim,
    return_state=True
)(encoder_embedding)

encoder_states = [state_h, state_c]

In [6]:
decoder_inputs = tf.keras.layers.Input(shape=(max_ger_len,))
decoder_embedding = tf.keras.layers.Embedding(
    ger_vocab_size,
    embedding_dim
)(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = tf.keras.layers.Dense(
    ger_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

In [7]:
model = tf.keras.Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Translation model created")

Translation model created


In [8]:
history = model.fit(
    [encoder_input, decoder_input],
    np.expand_dims(decoder_output, -1),
    epochs=300,
    batch_size=2,
    verbose=0
)

print("Training completed")
print("Final loss:", history.history["loss"][-1])
print("Final accuracy:", history.history["accuracy"][-1])

Training completed
Final loss: 0.00160291010979563
Final accuracy: 1.0


In [10]:
reverse_ger_index = {value: key for key, value in ger_tokenizer.word_index.items()}

def translate_sentence(sentence):
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding="post")

    start_token = ger_tokenizer.word_index["startseq"]

    decoder_seq = np.zeros((1, max_ger_len))
    decoder_seq[0, 0] = start_token

    prediction = model.predict([seq, decoder_seq], verbose=0)
    predicted_ids = np.argmax(prediction[0], axis=1)

    words = []

    for idx in predicted_ids:
        word = reverse_ger_index.get(idx, "")
        if word == "endseq":
            break
        if word not in ["startseq", ""]:
            words.append(word)

    return " ".join(words)

In [11]:
test_sentences = [
    "hello",
    "good morning",
    "thank you",
    "what is your name",
    "i am a student"
]

for sentence in test_sentences:
    print("English:", sentence)
    print("German:", translate_sentence(sentence))
    print()

English: hello
German: hallo

English: good morning
German: guten morgen

English: thank you
German: danke

English: what is your name
German: wie heisst du

English: i am a student
German: ich bin ein student

